# Start Trek — DQN

Agent DQN entraîné à poser le module de `LunarLander-v3`.

Objectif : score moyen ≥ 200 sur 100 épisodes consécutifs, sans exploration.

## Sommaire

1. Imports et configuration
2. Environnement
3. Raison de fin d'épisode
4. Boucle d'épisode et logs
5. Baseline : agent random

## 1. Imports et configuration

Toutes les seeds sont regroupées ici pour que les résultats soient reproductibles.

In [ ]:
import json
import time
from collections import Counter
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
ENV_ID = "LunarLander-v3"
SEED = 0
SEEDS = [0, 1, 2]

RESULTS_DIR = Path("results/dqn")
MODELS_DIR = Path("models")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)

## 2. Environnement

Deux façons de créer l'environnement :

- **entraînement / évaluation** : `render_mode=None`, aucun rendu, c'est beaucoup plus rapide ;
- **visualisation** : `render_mode="human"`, uniquement pour regarder l'agent jouer.

La taille de l'état et le nombre d'actions sont lus depuis l'environnement, pas écrits en dur.

In [ ]:
def make_env(render_mode=None):
    return gym.make(ENV_ID, render_mode=render_mode)


env = make_env()
OBS_DIM = env.observation_space.shape[0]
N_ACTIONS = env.action_space.n
MAX_STEPS = env.spec.max_episode_steps

print(f"Taille de l'état : {OBS_DIM}")
print(f"Nombre d'actions : {N_ACTIONS}")
print(f"Steps max par épisode : {MAX_STEPS}")

## 3. Raison de fin d'épisode

Gymnasium renvoie seulement `terminated` et `truncated`, pas la raison exacte. On la retrouve à partir de l'état interne de l'environnement (`env.unwrapped`), avec les mêmes conditions que le code source de LunarLander :

| Label | Type | Condition |
| --- | --- | --- |
| `crash` | terminated | le corps du module a touché le sol (`game_over`) |
| `out_of_viewport` | terminated | `abs(x) >= 1` |
| `at_rest` | terminated | le module ne bouge plus (`lander.awake == False`) |
| `truncated` | truncated | limite de steps atteinte |

`crash` est testé en premier : si le module touche le sol et sort de l'écran au même step, c'est le contact qui a déclenché la fin.

In [ ]:
END_REASONS = ["crash", "out_of_viewport", "at_rest", "truncated"]


def get_end_reason(env, state, terminated, truncated):
    if truncated and not terminated:
        return "truncated"
    lander_env = env.unwrapped
    if lander_env.game_over:
        return "crash"
    if abs(state[0]) >= 1.0:
        return "out_of_viewport"
    if not lander_env.lander.awake:
        return "at_rest"
    return "unknown"

## 4. Boucle d'épisode et logs

`run_episode` joue un épisode complet avec une politique donnée (une fonction `state -> action`) et renvoie un log :

| Champ | Contenu |
| --- | --- |
| `episode` | numéro de l'épisode |
| `reward` | reward totale |
| `length` | nombre de steps |
| `end_reason` | raison de fin (section 3) |
| `global_step` | nombre total de steps joués depuis le début |
| `wall_time` | temps écoulé depuis le début (secondes) |

Ce format est le même dans `ppo.ipynb`, pour pouvoir comparer les deux agents.

In [ ]:
def run_episode(env, policy, seed=None, episode=0, global_step=0, start_time=None):
    if start_time is None:
        start_time = time.time()
    state, _ = env.reset(seed=seed)
    total_reward = 0.0
    length = 0
    done = False

    while not done:
        action = policy(state)
        state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        length += 1
        done = terminated or truncated

    return {
        "episode": episode,
        "reward": float(total_reward),
        "length": length,
        "end_reason": get_end_reason(env, state, terminated, truncated),
        "global_step": global_step + length,
        "wall_time": time.time() - start_time,
    }


def summarize(logs):
    rewards = np.array([log["reward"] for log in logs])
    reasons = Counter(log["end_reason"] for log in logs)
    return {
        "n_episodes": len(logs),
        "mean": float(rewards.mean()),
        "std": float(rewards.std()),
        "min": float(rewards.min()),
        "max": float(rewards.max()),
        "end_reasons": {reason: reasons.get(reason, 0) for reason in END_REASONS},
    }

## 5. Baseline : agent random

Avant d'entraîner quoi que ce soit, on joue 100 épisodes en choisissant les actions au hasard. Ça donne un score de référence et ça vérifie que les logs marchent.

Chaque épisode `i` utilise la seed `SEED + i` : la baseline est identique à chaque exécution.

In [ ]:
N_BASELINE_EPISODES = 100

env.action_space.seed(SEED)
random_policy = lambda state: env.action_space.sample()

baseline_logs = []
global_step = 0
start_time = time.time()
for i in range(N_BASELINE_EPISODES):
    log = run_episode(env, random_policy, seed=SEED + i, episode=i,
                      global_step=global_step, start_time=start_time)
    global_step = log["global_step"]
    baseline_logs.append(log)

baseline = summarize(baseline_logs)
print(f"Score moyen : {baseline['mean']:.1f} ± {baseline['std']:.1f}")
print(f"Min / max   : {baseline['min']:.1f} / {baseline['max']:.1f}")
print(f"Raisons de fin : {baseline['end_reasons']}")

In [ ]:
rewards = [log["reward"] for log in baseline_logs]
reasons = baseline["end_reasons"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(rewards, bins=20, color="tab:gray", edgecolor="black")
axes[0].axvline(baseline["mean"], color="tab:red", linestyle="--", label=f"moyenne {baseline['mean']:.1f}")
axes[0].axvline(200, color="tab:green", linestyle="--", label="objectif 200")
axes[0].set_title("Agent random : distribution des scores")
axes[0].set_xlabel("Reward totale")
axes[0].set_ylabel("Nombre d'épisodes")
axes[0].legend()

axes[1].bar(reasons.keys(), reasons.values(), color="tab:gray", edgecolor="black")
axes[1].set_title("Agent random : raisons de fin")
axes[1].set_ylabel("Nombre d'épisodes")

plt.tight_layout()
plt.show()

In [ ]:
with open(RESULTS_DIR / "baseline_random.json", "w") as f:
    json.dump({"summary": baseline, "episodes": baseline_logs}, f, indent=2)

**Analyse :** _à compléter après exécution (score moyen, raison de fin dominante, ce que ça dit du problème)._